In [190]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Load Data 

In [191]:
data_path = Path("../data")

movie_path = f"{data_path}/refine_movies.csv"
rating_path = f"{data_path}/refine_ratings.csv"
tag_path = f"{data_path}/refine_tags.csv"
link_path = f"{data_path}/refine_links.csv"

In [192]:
movies_df = pd.read_csv(
    movie_path,
    usecols=['movieId', 'title','year','genres'],
    dtype={'title':str, 'year':str, 'movieId':int, 'genres':str})

ratings_df = pd.read_csv(
    rating_path,
    usecols=['userId','movieId','rating'],
    dtype={'userId':int, 'movieId':int, 'rating':float}
    )
tags_df = pd.read_csv(
    tag_path,
    usecols=['userId','movieId','tag'],
    dtype={'userId':int, 'movieId':int, 'tag':str})
links_df = pd.read_csv(link_path)

In [193]:
df_list = {
    "movies" : movies_df,
    "ratings" : ratings_df,
    "tags" : tags_df,
    "links_df" : links_df
}

# Data check

In [194]:
# genres to list
import ast
movies_df['genres'] = movies_df['genres'].apply(lambda x : ast.literal_eval(x))

movies_df['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9734                 [Action, Animation, Comedy, Fantasy]
9735                         [Animation, Comedy, Fantasy]
9736                                              [Drama]
9737                                  [Action, Animation]
9738                                             [Comedy]
Name: genres, Length: 9739, dtype: object

In [195]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [196]:
tags_df

,userId,movieId,tag
0,2,60756,['funny']
1,2,60756,"['Highly', 'quotable']"
2,2,60756,"['will', 'ferrell']"
3,2,89774,"['Boxing', 'story']"
4,2,89774,['MMA']
...,...,...,...
3678,606,7382,"['for', 'katie']"
3679,606,7936,['austere']
3680,610,3265,"['gun', 'fu']"
3681,610,3265,"['heroic', 'bloodshed']"


# Data Merge

## (1) rating centering
: 유저 bias 제거 . 분산도 함께 반영

### Z score normalization

In [197]:
temp_rating = ratings_df.copy()

func = lambda rating : (rating - rating.mean()) / rating.std()

temp_rating['centered_rating'] = ratings_df.groupby(by='userId')['rating'].transform(func) # apply 는 원래 구조를 변경, transform 는 원래 DF 구조를 유지

temp_rating['centered_rating'] = temp_rating['centered_rating'].fillna(0) # 평가가 1개인 경우 std=0 이라서 NaN 값이 출력 → 0 으로 수정

ratings_df = temp_rating

In [198]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movieId', how='left') # on : merge key
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017)


# zero ratings movie check

In [199]:
movie_ratings_df.isnull().sum()

userId             0
movieId            0
rating             0
centered_rating    0
title              0
genres             0
year               0
dtype: int64

In [200]:
no_ratings_movieId = movie_ratings_df.loc[movie_ratings_df.isnull().any(axis=1)]['movieId'].values
print("평가가 존재하지 않는 영화 ID 갯수: ",len(no_ratings_movieId))
no_ratings_movieId

평가가 존재하지 않는 영화 ID 갯수:  0


array([], dtype=int64)

In [201]:
movie_ratings_df[movie_ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centered_rating,title,genres,year


In [202]:
# double check (ratings_df 에도 존재하지 않는지 확인)
ratings_df[ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centered_rating


# Item User Matrix

In [203]:
item_user_matrix = movie_ratings_df.pivot_table(values='centered_rating', index='movieId', columns='userId')

no_centered_item_user_matrix = movie_ratings_df.pivot_table(values='rating', index='movieId', columns='userId')

In [204]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,-0.457947,NaN,NaN,NaN,0.367146,NaN,0.954981,NaN,NaN,NaN,...,-0.935911,NaN,0.426517,-0.644222,0.964425,-1.598352,0.221511,-0.587601,-0.6003,1.52952
2,NaN,NaN,NaN,NaN,NaN,0.595275,NaN,0.437643,NaN,NaN,...,NaN,0.621013,NaN,2.040036,0.353715,NaN,NaN,-1.050881,NaN,NaN
3,-0.457947,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.050881,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,-0.580300,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.644222,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 유저별/영화별 평균평점

In [205]:
user_rating_mean = ratings_df.groupby(by='userId')['rating'].apply(lambda x: x.mean())
user_rating_mean

userId
1      4.366379
2      3.948276
3      2.435897
4      3.555556
5      3.636364
         ...   
606    3.657399
607    3.786096
608    3.134176
609    3.270270
610    3.688556
Name: rating, Length: 610, dtype: float64

In [206]:
movie_rating_mean = ratings_df.groupby(by='movieId')['rating'].apply(lambda x: x.mean())
movie_rating_mean


movieId
1         3.920930
2         3.431818
3         3.259615
4         2.357143
5         3.071429
            ...   
193581    4.000000
193583    3.500000
193585    3.500000
193587    3.500000
193609    4.000000
Name: rating, Length: 9721, dtype: float64

In [207]:
temp_df = movies_df.set_index('movieId')
temp_df = temp_df.join(movie_rating_mean.rename("avg_rating"))
movies_df = temp_df.reset_index()

In [208]:
movies_df

,movieId,title,genres,year,avg_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429
...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000
9736,193585,Flint (2017),[Drama],(2017),3.500000
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000


## (1) 영화당 평균 평점 갯수 계산
- a. 영화 평점 갯수 1개, 5.0
  
- b. 영화 평점 갯수 100개, 4.5

- 영화 평점 갯수가 적은 영화에 대해 평균 평점 조정

In [209]:
print("평균 평점 갯수 :" ,round(item_user_matrix.apply(lambda x: x.count(), axis=1).mean()))
ratings_count = item_user_matrix.apply(lambda x: x.count(), axis=1)
ratings_count

평균 평점 갯수 : 10


movieId
1         215
2         110
3          52
4           7
5          49
         ... 
193581      1
193583      1
193585      1
193587      1
193609      1
Length: 9721, dtype: int64

In [210]:
ratings_count.sort_values(ascending=False)

movieId
356       329
318       317
296       307
593       279
2571      278
         ... 
58351       1
4089        1
58492       1
4083        1
193609      1
Length: 9721, dtype: int64

In [211]:
threshold = ratings_count.sort_values().quantile(0.8) # 상위 10% 평점 갯수
threshold

12.0

In [212]:
temp_df = movies_df.set_index(keys='movieId').copy()
temp_df = temp_df.join(ratings_count.rename('count'))
movies_df = temp_df.reset_index()

In [213]:
movies_df = movies_df.dropna(subset='count') # 리뷰 0개인 영화 제거
movies_df['count'] = movies_df['count'].astype('int64')

movies_df

,movieId,title,genres,year,avg_rating,count
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,215
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,110
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,52
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,7
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,49
...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,1
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,1
9736,193585,Flint (2017),[Drama],(2017),3.500000,1
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,1


In [214]:
nrating_weighted_movieid = movies_df[movies_df['count'] > threshold]['movieId']
print("상향 조정할 영화 갯수 : ",len(nrating_weighted_movieid))

movies_df

상향 조정할 영화 갯수 :  1849


,movieId,title,genres,year,avg_rating,count
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,215
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,110
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,52
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,7
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,49
...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,1
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,1
9736,193585,Flint (2017),[Drama],(2017),3.500000,1
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,1


In [215]:
movies_mean = movies_df['avg_rating'].mean() # 전체 영화 평균

def weighted_rating(df, threshold=threshold, mean=movies_mean):
    c = df['count']
    r = df['avg_rating']
    return 0.3*( c / (c+threshold)* r ) + ( c / (c +threshold) * mean)

movies_df['weighted_rating'] = movies_df.apply(weighted_rating, axis=1) # 리뷰가 많은 영화 → avg_rating 을 높여준다.
movies_df

,movieId,title,genres,year,avg_rating,count,weighted_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,215,4.204024
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,110,3.869776
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,52,3.445221
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,7,1.462459
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,49,3.360771
...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,1,0.343261
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,1,0.331722
9736,193585,Flint (2017),[Drama],(2017),3.500000,1,0.331722
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,1,0.331722


In [216]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017)


In [217]:
temp_map = movies_df.set_index('movieId')['weighted_rating']

movie_ratings_df['weighted_centered_rating'] = movie_ratings_df['rating'] - movie_ratings_df['movieId'].map(temp_map)
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_centered_rating
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),-0.204024
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),0.554779
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995),0.021811
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995),0.793652
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995),0.718162
...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017),2.579204
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017),3.340173
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017),2.582007
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017),1.928116


In [218]:
weighted_item_user_matrix = movie_ratings_df.pivot_table(values='weighted_centered_rating', index='movieId', columns='userId')
weighted_item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,-0.204024,NaN,NaN,NaN,-0.204024,NaN,0.295976,NaN,NaN,NaN,...,-0.204024,NaN,-0.204024,-1.204024,-0.204024,-1.704024,-0.204024,-1.704024,-1.204024,0.795976
2,NaN,NaN,NaN,NaN,NaN,0.130224,NaN,0.130224,NaN,NaN,...,NaN,0.130224,NaN,1.130224,-0.369776,NaN,NaN,-1.869776,NaN,NaN
3,0.554779,NaN,NaN,NaN,NaN,1.554779,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.445221,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,1.537541,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,1.639229,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.360771,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 유저별 장르 평균 점수

In [219]:
genres_list = movies_df['genres'].explode().unique()

In [220]:
movies_df

,movieId,title,genres,year,avg_rating,count,weighted_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,215,4.204024
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,110,3.869776
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,52,3.445221
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,7,1.462459
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,49,3.360771
...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,1,0.343261
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,1,0.331722
9736,193585,Flint (2017),[Drama],(2017),3.500000,1,0.331722
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,1,0.331722


In [221]:
def calc_user_genre_mean(group):
    genres_totals = {genre : 0 for genre in genres_list}
    genres_count = {genre : 0 for genre in genres_list}
    
    for _, row in group.iterrows():
        rating = row['centered_rating']
        movie_genres = row['genres']
        for genre in movie_genres:
            genres_totals[genre] += rating
            genres_count[genre] += 1
    genres_avg = {}
    for genre in genres_list:
        if genres_count[genre] > 0 :
            genres_avg[genre] = genres_totals[genre]/genres_count[genre]
        else:
            genres_avg[genre] = pd.NA
    return pd.Series(genres_avg)

In [222]:
def improved_calc_genre_mean(movies_ratings):
    temp_df = movies_ratings.explode('genres')
    genres_means = temp_df.groupby(['userId','genres'])['centered_rating'].mean()
    user_genre_avg = genres_means.unstack()
    return user_genre_avg


In [223]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_centered_rating
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),-0.204024
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),0.554779
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995),0.021811
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995),0.793652
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995),0.718162
...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017),2.579204
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017),3.340173
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017),2.582007
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017),1.928116


In [224]:
user_genre_avg = movie_ratings_df.groupby('userId').apply(calc_user_genre_mean)

/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_19504/2273720666.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_genre_avg = movie_ratings_df.groupby('userId').apply(calc_user_genre_mean)


In [225]:
user_genre_avg

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
userId,,,,,,,,,,,,,,,,,,,
1,0.027318,0.404071,0.226536,-0.111582,-0.085629,-0.073354,0.203778,-0.055193,-0.013529,-0.276139,-1.119672,-0.249626,-0.176714,0.167016,0.394275,<NA>,<NA>,-0.100825,0.791978
2,0.271086,<NA>,<NA>,0.064205,<NA>,0.684849,-0.081829,0.007782,-0.184053,-0.308182,-1.177084,0.064205,-0.090956,0.684849,<NA>,0.477967,-0.246118,-0.55644,<NA>
3,0.030662,-0.925982,-0.925982,-0.686821,0.449193,-0.925982,-0.806402,0.54315,-0.925982,0.816476,1.076991,1.226467,0.843809,-0.925982,-0.925982,<NA>,<NA>,<NA>,<NA>
4,0.0758,0.338185,0.186002,-0.034957,0.097896,-0.134108,-0.054955,-0.179238,0.197275,-0.002225,0.528415,-0.058815,-0.549551,0.012078,0.338185,0.338185,-0.422732,0.186002,0.338185
5,-0.390093,0.703697,0.47933,-0.171335,0.511382,-0.550719,0.165216,-0.530322,0.198871,-0.081588,-0.642506,0.367146,-1.147331,-0.305955,0.771007,<NA>,0.030596,-0.642506,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.212669,0.07856,-0.287824,-0.127159,-0.082115,0.114303,0.180310,-0.660928,-0.004507,-0.182668,-0.429825,0.184789,-0.138702,0.186307,0.096494,0.19693,-0.821547,-0.339218,0.214192
607,-0.33079,-0.468865,-0.378026,-0.475141,-0.222302,-0.278416,0.234140,-0.066146,0.02974,0.340346,0.339861,0.891582,-0.555162,0.394105,-0.192715,<NA>,1.257075,0.221511,<NA>
608,0.080443,-0.014819,-0.624453,-0.368359,-0.124322,-0.229215,0.281048,0.181744,0.443672,0.372944,0.171795,0.385957,0.150317,0.412107,-0.348942,-0.124322,0.802237,-0.461252,0.570598


# 영화별 장르 매트릭스 계산

In [226]:
genres_list

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

In [227]:
movies_df

,movieId,title,genres,year,avg_rating,count,weighted_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,215,4.204024
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,110,3.869776
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,52,3.445221
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,7,1.462459
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,49,3.360771
...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,1,0.343261
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,1,0.331722
9736,193585,Flint (2017),[Drama],(2017),3.500000,1,0.331722
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,1,0.331722


In [228]:
movie_genre_matrix = pd.DataFrame(0, index=movies_df['movieId'], columns=genres_list)
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
movieId,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
193583,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
193585,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [229]:
for genre in genres_list:
    movie_genre_matrix[genre] = movies_df.set_index('movieId')['genres'].apply(lambda x : 1 if genre in x else 0) # apply 적용을 위해 movies_df 의 인덱스 movieId 로 설정
    
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
movieId,,,,,,,,,,,,,,,,,,,
1,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
193583,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
193585,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [230]:
genre_count = movie_genre_matrix.sum(axis=1)

movie_genre_matrix = movie_genre_matrix.div(genre_count.replace(0,1), axis=0) # 장르가 많은 영화의 점수가 많이 반영되지 않도록 정규화

movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
movieId,,,,,,,,,,,,,,,,,,,
1,0.200000,0.200000,0.200000,0.200000,0.200000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.333333,0.000000,0.333333,0.000000,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.000000,0.000000,0.333333,0.000000,0.333333,0.333333,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.000000,0.250000,0.000000,0.250000,0.250000,0.000000,0.000000,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193583,0.000000,0.333333,0.000000,0.333333,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
193585,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [231]:
print(user_genre_avg.shape)
print(movie_genre_matrix.shape)

(610, 19)
(9721, 19)


# 예측 평점 행렬 계산

user_genre_avg 와 movie_genre_matrix 내적
- 결과 shape 사용자 수 x 영화 수

In [232]:
pred_ratings_use_genre = user_genre_avg.fillna(0).dot(movie_genre_matrix.T)
pred_ratings_use_genre

/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_19504/2084021829.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pred_ratings_use_genre = user_genre_avg.fillna(0).dot(movie_genre_matrix.T)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.092143,0.056075,-0.092468,0.006281,-0.111582,-0.114954,-0.092468,0.126927,-0.055193,-0.101338,...,0.015146,0.303924,0.046098,0.404071,0.000000,0.037917,0.068953,0.203778,0.174439,-0.111582
2,0.067058,0.090362,0.374527,0.222408,0.064205,-0.161484,0.374527,0.135543,0.007782,-0.009771,...,-0.004742,-0.040915,-0.008812,0.000000,0.477967,0.017997,0.021402,-0.081829,0.003891,0.064205
3,-0.411786,-0.148709,-0.806402,-0.806402,-0.686821,0.144548,-0.806402,-0.447660,0.543150,0.463429,...,-0.056461,-0.866192,-0.746612,-0.925982,0.000000,-0.155115,-0.387870,-0.806402,-0.191416,-0.686821
4,0.132585,0.119899,-0.084532,-0.074673,-0.034957,0.005271,-0.084532,0.130901,-0.179238,-0.035221,...,-0.106390,0.141615,-0.044956,0.338185,0.338185,0.055472,0.133708,-0.054955,0.079474,-0.034957
5,0.226596,0.200206,-0.361027,-0.185613,-0.171335,-0.137680,-0.361027,0.044618,-0.530322,-0.334001,...,-0.286323,0.434456,-0.003060,0.703697,0.000000,0.128355,0.347915,0.165216,0.086687,-0.171335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.126241,-0.194203,-0.006428,0.055818,-0.127159,-0.282701,-0.006428,-0.250247,-0.660928,-0.352088,...,-0.212057,0.129435,0.026576,0.078560,0.196930,-0.197910,-0.043571,0.180310,-0.291184,-0.127159
607,-0.375025,-0.310373,-0.376779,-0.173139,-0.475141,0.101314,-0.376779,-0.354408,-0.066146,-0.018863,...,-0.391329,-0.117363,-0.120501,-0.468865,0.000000,-0.308114,-0.388769,0.234140,-0.267505,-0.475141
608,-0.210302,-0.222777,-0.298787,-0.105509,-0.368359,0.332786,-0.298787,-0.272005,0.181744,0.211710,...,-0.012779,0.133114,-0.043656,-0.014819,-0.124322,-0.081439,-0.169167,0.281048,0.083462,-0.368359


# fill NaN value in movie-user matrix

## (1) sim matrix

In [233]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,-0.457947,NaN,NaN,NaN,0.367146,NaN,0.954981,NaN,NaN,NaN,...,-0.935911,NaN,0.426517,-0.644222,0.964425,-1.598352,0.221511,-0.587601,-0.6003,1.52952
2,NaN,NaN,NaN,NaN,NaN,0.595275,NaN,0.437643,NaN,NaN,...,NaN,0.621013,NaN,2.040036,0.353715,NaN,NaN,-1.050881,NaN,NaN
3,-0.457947,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.050881,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,-0.580300,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.644222,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [234]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim_fill_zero = cosine_similarity(item_user_matrix.fillna(0)) # 평가가 유사한것이 유사도의 기준 

cos_sim_zero_df = pd.DataFrame(cos_sim_fill_zero, index=item_user_matrix.index, columns=item_user_matrix.index)
cos_sim_zero_df.head(5)

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.003913,0.042058,-0.040181,-0.149957,0.116043,-0.027761,0.001167,-0.110610,-0.015561,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.003913,1.000000,0.055780,-0.092855,0.079447,-0.000128,0.023389,0.027081,-0.005871,-0.098907,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.042058,0.055780,1.000000,-0.042740,0.146563,-0.036561,0.083836,-0.064968,-0.010080,-0.119355,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.040181,-0.092855,-0.042740,1.000000,0.016135,-0.022556,0.022205,0.034538,0.000000,0.128928,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,-0.149957,0.079447,0.146563,0.016135,1.000000,-0.043216,0.195040,0.003483,0.071630,0.019909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# (2) genre 이용 sim matrix

In [235]:
pred_ratings_use_genre.T[pred_ratings_use_genre.T.index==1]

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.092143,0.067058,-0.411786,0.132585,0.226596,0.240394,-0.002281,0.284153,0.654182,0.232814,...,0.137212,0.052801,-0.389747,-0.017829,-0.055557,-0.126241,-0.375025,-0.210302,-0.384535,0.032743


In [236]:
pred_ratings_use_genre.T

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.092143,0.067058,-0.411786,0.132585,0.226596,0.240394,-0.002281,0.284153,0.654182,0.232814,...,0.137212,0.052801,-0.389747,-0.017829,-0.055557,-0.126241,-0.375025,-0.210302,-0.384535,0.032743
2,0.056075,0.090362,-0.148709,0.119899,0.200206,0.222657,-0.027767,0.110405,0.790394,0.224901,...,0.177435,-0.087448,-0.227352,0.100047,-0.002992,-0.194203,-0.310373,-0.222777,-0.452226,-0.044776
3,-0.092468,0.374527,-0.806402,-0.084532,-0.361027,-0.001703,-0.243406,-0.226571,0.122431,0.017867,...,-0.139026,-0.125339,-0.113648,-0.434308,0.102673,-0.006428,-0.376779,-0.298787,-0.060888,0.049639
4,0.006281,0.222408,-0.806402,-0.074673,-0.185613,0.046145,-0.187011,-0.077339,0.125550,-0.023753,...,-0.081375,-0.117119,-0.046285,-0.286326,0.031088,0.055818,-0.173139,-0.105509,0.032076,0.105474
5,-0.111582,0.064205,-0.686821,-0.034957,-0.171335,-0.145244,-0.050390,-0.376555,0.318891,-0.010843,...,-0.166505,-0.094667,-0.176258,-0.320260,0.058652,-0.127159,-0.475141,-0.368359,0.034303,0.049669
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.037917,0.017997,-0.155115,0.055472,0.128355,0.180692,-0.007878,0.126962,0.539908,0.204002,...,0.060255,0.259522,-0.378701,-0.069919,-0.080639,-0.197910,-0.308114,-0.081439,-0.391170,0.020914
193583,0.068953,0.021402,-0.387870,0.133708,0.347915,0.195568,-0.017411,0.251948,0.755468,0.209223,...,0.124949,0.285082,-0.409156,-0.158095,-0.113312,-0.043571,-0.388769,-0.169167,-0.388766,0.062088
193585,0.203778,-0.081829,-0.806402,-0.054955,0.165216,0.141839,-0.074221,0.221125,0.131787,-0.106995,...,0.033928,-0.100681,0.088440,0.009636,-0.112081,0.180310,0.234140,0.281048,0.218004,0.217143


# (3) Year

In [237]:
movies_year = movies_df['year'].str.strip(to_strip="()")

## 연도 추가 전처리

In [238]:
movies_year[movies_year.apply(lambda x : len(x) > 4)]
movies_df.loc[9515, 'year'] = "(2006)"
movies_df.loc[9515]

movieId                                           171749
title                  Death Note: Desu nôto (2006–2007)
genres             [Animation, Mystery, Sci-Fi, Fantasy]
year                                              (2006)
avg_rating                                           5.0
count                                                  1
weighted_rating                                 0.366338
Name: 9515, dtype: object

In [239]:
movies_df

,movieId,title,genres,year,avg_rating,count,weighted_rating
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),3.920930,215,4.204024
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995),3.431818,110,3.869776
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995),3.259615,52,3.445221
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995),2.357143,7,1.462459
4,5,Father of the Bride Part II (1995),[Comedy],(1995),3.071429,49,3.360771
...,...,...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017),4.000000,1,0.343261
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017),3.500000,1,0.331722
9736,193585,Flint (2017),[Drama],(2017),3.500000,1,0.331722
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018),3.500000,1,0.331722


## 연도의 괄호 제거

In [240]:
movies_year = movies_df.set_index('movieId')['year'].str.strip(to_strip="()")

In [241]:
movies_year

movieId
1         1995
2         1995
3         1995
4         1995
5         1995
          ... 
193581    2017
193583    2017
193585    2017
193587    2018
193609    1991
Name: year, Length: 9721, dtype: object

In [242]:
movie_ratings_df['year'] = movie_ratings_df['movieId'].map(movies_year.to_dict())

movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_centered_rating
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,-0.204024
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.554779
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.021811
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,0.793652
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.718162
...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,2.579204
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,3.340173
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,2.582007
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.928116


### 연도가 조사되지 않은 영화 확인
- 전처리 체크

In [243]:
print("연도 영화하지 않는 영화 : " ,movie_ratings_df['year'].isnull().sum(), ", 전체 :", len(movie_ratings_df))

연도 영화하지 않는 영화 :  0 , 전체 : 100836


## 각 연도별 평균 부여 점수 계산

In [244]:
movie_ratings_df.groupby(['userId','year'])['rating'].mean()

userId  year
1       1922    4.000000
        1930    5.000000
        1931    4.000000
        1933    4.500000
        1938    5.000000
                  ...   
610     2013    3.626866
        2014    3.791667
        2015    3.632653
        2016    3.741935
        2017    4.400000
Name: rating, Length: 17440, dtype: float64

In [245]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_centered_rating
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,-0.204024
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.554779
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.021811
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,0.793652
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.718162
...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,2.579204
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,3.340173
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,2.582007
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.928116


In [246]:
user_year_matrix = movie_ratings_df.groupby(['userId','year'])['rating'].mean().unstack()

user_year_matrix

year,1902,1903,1908,1915,1916,1917,1919,1920,1921,1922,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.750000,4.083333,3.750000,3.500000,5.000000,3.000000,4.500000,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,...,3.500000,4.000000,NaN,3.375000,NaN,NaN,NaN,NaN,NaN,NaN
607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 영화에 대한 matrix 계산

In [247]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_centered_rating
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,-0.204024
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.554779
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.021811
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,0.793652
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.718162
...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,2.579204
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,3.340173
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,2.582007
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.928116


In [248]:
movie_year_matrix = pd.get_dummies(movies_year).astype('int64')

movie_year_matrix

,1902,1903,1908,1915,1916,1917,1919,1920,1921,1922,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
movieId,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
193583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
193585,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [249]:
user_year_matrix.count(axis=1)

userId
1      53
2      15
3      23
4      47
5      10
       ..
606    85
607    40
608    50
609     9
610    63
Length: 610, dtype: int64

In [250]:
print(movie_year_matrix.shape)
print(user_year_matrix.shape)

(9721, 106)
(610, 106)


In [251]:
user_year_matrix = user_year_matrix.apply(lambda x : (x - x.mean())/x.std())
user_year_matrix

year,1902,1903,1908,1915,1916,1917,1919,1920,1921,1922,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.597425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.003776,0.351061,0.109923,-0.336696,1.854890,-1.098558,1.024749,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-4.137905,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.21356,NaN,...,-0.321786,0.237316,NaN,-0.498609,NaN,NaN,NaN,NaN,NaN,NaN
607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [252]:
pred_ratings_use_year = user_year_matrix.fillna(0).dot(movie_year_matrix.T)
pred_ratings_use_year

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,1.364919,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,0.655538
2,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,0.512715,...,0.351061,1.854890,-1.098558,1.024749,1.024749,0.00000,0.00000,0.00000,0.0,0.000000
3,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,-4.174405,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-3.829297
4,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,-0.960379,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-0.514419
5,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,0.066323,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,0.265552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,-0.006216,...,0.237316,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-0.074113
607,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,0.177921,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,0.499543
608,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,-1.154133,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0,-1.799078


In [253]:
pred_ratings_use_genre

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,0.092143,0.056075,-0.092468,0.006281,-0.111582,-0.114954,-0.092468,0.126927,-0.055193,-0.101338,...,0.015146,0.303924,0.046098,0.404071,0.000000,0.037917,0.068953,0.203778,0.174439,-0.111582
2,0.067058,0.090362,0.374527,0.222408,0.064205,-0.161484,0.374527,0.135543,0.007782,-0.009771,...,-0.004742,-0.040915,-0.008812,0.000000,0.477967,0.017997,0.021402,-0.081829,0.003891,0.064205
3,-0.411786,-0.148709,-0.806402,-0.806402,-0.686821,0.144548,-0.806402,-0.447660,0.543150,0.463429,...,-0.056461,-0.866192,-0.746612,-0.925982,0.000000,-0.155115,-0.387870,-0.806402,-0.191416,-0.686821
4,0.132585,0.119899,-0.084532,-0.074673,-0.034957,0.005271,-0.084532,0.130901,-0.179238,-0.035221,...,-0.106390,0.141615,-0.044956,0.338185,0.338185,0.055472,0.133708,-0.054955,0.079474,-0.034957
5,0.226596,0.200206,-0.361027,-0.185613,-0.171335,-0.137680,-0.361027,0.044618,-0.530322,-0.334001,...,-0.286323,0.434456,-0.003060,0.703697,0.000000,0.128355,0.347915,0.165216,0.086687,-0.171335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.126241,-0.194203,-0.006428,0.055818,-0.127159,-0.282701,-0.006428,-0.250247,-0.660928,-0.352088,...,-0.212057,0.129435,0.026576,0.078560,0.196930,-0.197910,-0.043571,0.180310,-0.291184,-0.127159
607,-0.375025,-0.310373,-0.376779,-0.173139,-0.475141,0.101314,-0.376779,-0.354408,-0.066146,-0.018863,...,-0.391329,-0.117363,-0.120501,-0.468865,0.000000,-0.308114,-0.388769,0.234140,-0.267505,-0.475141
608,-0.210302,-0.222777,-0.298787,-0.105509,-0.368359,0.332786,-0.298787,-0.272005,0.181744,0.211710,...,-0.012779,0.133114,-0.043656,-0.014819,-0.124322,-0.081439,-0.169167,0.281048,0.083462,-0.368359


In [254]:
pred_ratings_use_genre_year = pred_ratings_use_genre.add(pred_ratings_use_year)
pred_ratings_use_genre_year

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,1.457062,1.420994,1.272451,1.371200,1.253337,1.249965,1.272451,1.491846,1.309726,1.263581,...,0.015146,0.303924,0.046098,0.404071,0.000000,0.037917,0.068953,0.203778,0.174439,0.543956
2,0.579774,0.603077,0.887242,0.735123,0.576920,0.351231,0.887242,0.648258,0.520498,0.502944,...,0.346319,1.813975,-1.107371,1.024749,1.502717,0.017997,0.021402,-0.081829,0.003891,0.064205
3,-4.586191,-4.323114,-4.980807,-4.980807,-4.861226,-4.029857,-4.980807,-4.622065,-3.631255,-3.710975,...,-0.056461,-0.866192,-0.746612,-0.925982,0.000000,-0.155115,-0.387870,-0.806402,-0.191416,-4.516119
4,-0.827794,-0.840480,-1.044912,-1.035053,-0.995336,-0.955109,-1.044912,-0.829478,-1.139618,-0.995600,...,-0.106390,0.141615,-0.044956,0.338185,0.338185,0.055472,0.133708,-0.054955,0.079474,-0.549376
5,0.292919,0.266529,-0.294704,-0.119290,-0.105012,-0.071357,-0.294704,0.110942,-0.463999,-0.267678,...,-0.286323,0.434456,-0.003060,0.703697,0.000000,0.128355,0.347915,0.165216,0.086687,0.094217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.132457,-0.200418,-0.012643,0.049603,-0.133375,-0.288917,-0.012643,-0.256462,-0.667143,-0.358304,...,0.025259,0.129435,0.026576,0.078560,0.196930,-0.197910,-0.043571,0.180310,-0.291184,-0.201272
607,-0.197104,-0.132452,-0.198858,0.004782,-0.297220,0.279235,-0.198858,-0.176487,0.111776,0.159058,...,-0.391329,-0.117363,-0.120501,-0.468865,0.000000,-0.308114,-0.388769,0.234140,-0.267505,0.024402
608,-1.364435,-1.376910,-1.452920,-1.259642,-1.522492,-0.821346,-1.452920,-1.426138,-0.972389,-0.942423,...,-0.012779,0.133114,-0.043656,-0.014819,-0.124322,-0.081439,-0.169167,0.281048,0.083462,-2.167437


In [255]:
cos_sim_genre_year = cosine_similarity(pred_ratings_use_genre_year.T)

genre_year_sim_matrix = pd.DataFrame(cos_sim_genre_year, movies_df['movieId'], movies_df['movieId'])

In [256]:
genre_year_sim_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.992201,0.937751,0.939113,0.942758,0.900123,0.937751,0.982575,0.893879,0.933102,...,0.250952,0.284705,0.216505,0.264727,0.082266,0.204666,0.243542,0.042353,0.194310,0.371159
2,0.992201,1.000000,0.917556,0.920493,0.920617,0.889105,0.917556,0.985755,0.889646,0.930949,...,0.242261,0.254766,0.199724,0.222961,0.076510,0.198745,0.232152,0.031141,0.164454,0.348095
3,0.937751,0.917556,1.000000,0.991114,0.976337,0.905018,1.000000,0.908562,0.889149,0.918604,...,0.200273,0.185155,0.255940,0.084693,0.067052,0.063976,0.089834,0.071636,0.005915,0.375759
4,0.939113,0.920493,0.991114,1.000000,0.965673,0.930860,0.991114,0.911524,0.900818,0.934912,...,0.178628,0.198690,0.260956,0.071543,0.074925,0.032585,0.055232,0.140303,-0.015937,0.354718
5,0.942758,0.920617,0.976337,0.965673,1.000000,0.900568,0.976337,0.909629,0.887640,0.914038,...,0.219389,0.165997,0.270785,0.090914,0.068291,0.100089,0.127486,0.027023,0.023113,0.403320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.204666,0.198745,0.063976,0.032585,0.100089,0.038372,0.063976,0.175210,0.110720,0.082147,...,0.405170,0.420108,0.207357,0.592009,0.252292,1.000000,0.965138,0.453875,0.610263,0.114300
193583,0.243542,0.232152,0.089834,0.055232,0.127486,0.003399,0.089834,0.197103,0.039002,0.046048,...,0.362006,0.462037,0.221197,0.648677,0.224709,0.965138,1.000000,0.408236,0.576274,0.138270
193585,0.042353,0.031141,0.071636,0.140303,0.027023,0.114799,0.071636,0.034465,0.051231,0.067910,...,0.112636,0.328660,0.320669,0.152490,0.264391,0.453875,0.408236,1.000000,0.023796,-0.038360


In [257]:
genre_year_sim_matrix.iloc[0].sort_values(ascending=False)

movieId
1        1.000000
60       0.992201
2        0.992201
243      0.991192
258      0.990331
           ...   
1068    -0.082698
25940   -0.084040
2203    -0.093570
7333    -0.093570
2066    -0.098503
Name: 1, Length: 9721, dtype: float64

# 코드 정리

### 1. 데이터 load

### 2. movie-ratings df

ratings 과 movies 를 merge

### 3. item-user matrix

movie-ratings df 에 pivot 적용

### 4. movie-xxx matrix 계산

### 5.  dot 연산을 통한 movie-movie matrix

### 6. cos sim 계산해 영화 추천

In [258]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_centered_rating
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,-0.204024
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.554779
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.021811
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,0.793652
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.718162
...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,2.579204
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,3.340173
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,2.582007
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.928116


In [259]:
### eval 에 필요한 data
x_test = ratings_df.copy() # movie_ratings_df(expanded version of ratings_df) → x_test

y_test = ratings_df['userId']

# sim_matrix

user_rating_mean

item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,-0.457947,NaN,NaN,NaN,0.367146,NaN,0.954981,NaN,NaN,NaN,...,-0.935911,NaN,0.426517,-0.644222,0.964425,-1.598352,0.221511,-0.587601,-0.6003,1.52952
2,NaN,NaN,NaN,NaN,NaN,0.595275,NaN,0.437643,NaN,NaN,...,NaN,0.621013,NaN,2.040036,0.353715,NaN,NaN,-1.050881,NaN,NaN
3,-0.457947,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.050881,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,-0.580300,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.644222,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [260]:
from sklearn.model_selection import train_test_split

In [261]:
movie_ratings_df

,userId,movieId,rating,centered_rating,title,genres,year,weighted_centered_rating
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995,-0.204024
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995,0.554779
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995,0.021811
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995,0.793652
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995,0.718162
...,...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",2017,2.579204
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",2017,3.340173
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],2017,2.582007
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",2017,1.928116


In [262]:
movie_ratings_df # train-test-split → x_test

x_train, x_test, y_train, y_test = train_test_split(movie_ratings_df, movie_ratings_df['userId'], stratify=movie_ratings_df['userId'], test_size=0.2, random_state=42)

In [263]:
movieid_idx = x_train.pivot_table(values="centered_rating", index='movieId', columns='userId').index # x_train 에서 사용한 movieid 추출

### make matrix using x_test

- 개선사항 : user_genre_mean 속도 개선

In [264]:
def get_rating_item_user_matrix(movie_ratings_df, is_centered=False):
    value = "weighted_centered_rating" if is_centered==True else "rating"
    item_user_matrix = movie_ratings_df.pivot_table(values=value, index='movieId', columns='userId') # 영화별 유저의 weighted mean centering 평점 matrix
    return item_user_matrix.fillna(0)
    
    
def get_genre_item_user_matrix(movie_ratings_df, is_centered=False):
    user_genre_avg = improved_calc_genre_mean(movie_ratings_df)
    
    movie_genre_matrix = pd.DataFrame(0, index=movieid_idx, columns=genres_list)
    for genre in genres_list:
        movie_genre_matrix[genre] = movies_df.set_index('movieId')['genres'].apply(lambda x : 1 if genre in x else 0) # apply 적용을 위해 movies_df 의 인덱스 movieId 로 설정
    genre_count = movie_genre_matrix.sum(axis=1)
    movie_genre_matrix = movie_genre_matrix.div(genre_count.replace(0,1), axis=0)
    
    pred_ratings_matrix = user_genre_avg.fillna(0).dot(movie_genre_matrix.T)
    return pred_ratings_matrix.T
    
def get_year_item_user_matrix(movie_ratings_df):
    user_year_matrix = movie_ratings_df.groupby(['userId','year'])['rating'].mean().unstack()
    
    temp_df = movies_df.set_index('movieId').loc[movieid_idx]
    movies_year = temp_df['year'].str.strip(to_strip="()")
    movie_year_matrix = pd.get_dummies(movies_year).astype('int64')
    user_year_matrix = user_year_matrix.apply(lambda x: (x-x.mean())/ x.std())
    
    pred_ratings_matrix = user_year_matrix.fillna(0).dot(movie_year_matrix.T)
    return pred_ratings_matrix.T
    

In [265]:
mat1 =cosine_similarity(get_rating_item_user_matrix(x_train, True)) # mavie_ratings_df → x_test
mat1 = pd.DataFrame(mat1, index=movieid_idx, columns=movieid_idx)
mat1

movieId,1,2,3,4,5,6,7,8,9,10,...,190219,190221,193565,193567,193571,193579,193581,193583,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.141696,0.068684,0.010160,0.071879,0.018974,0.086624,0.039328,0.035852,0.061432,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.141696,1.000000,0.300517,0.002619,0.180489,0.076303,0.089229,-0.040199,0.032510,0.086337,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.068684,0.300517,1.000000,0.104378,0.175675,0.271763,0.116891,0.228177,0.106331,0.042163,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.010160,0.002619,0.104378,1.000000,-0.030826,0.002964,-0.077512,0.152303,0.000000,-0.097297,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.071879,0.180489,0.175675,-0.030826,1.000000,0.038894,0.165340,0.137238,0.028797,0.142890,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193579,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
193581,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
193583,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0


In [266]:
mat2 = cosine_similarity(get_genre_item_user_matrix(x_train))
mat2 = pd.DataFrame(mat2, index=movieid_idx, columns=movieid_idx)
mat2

movieId,1,2,3,4,5,6,7,8,9,10,...,190219,190221,193565,193567,193571,193579,193581,193583,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.935714,0.335756,0.223952,0.402489,-0.134048,0.335756,0.871973,0.030166,0.190508,...,0.774636,-0.021795,0.633770,0.619393,0.205676,-0.021795,0.905816,0.930034,0.689953,0.402489
2,0.935714,1.000000,0.205458,0.101578,0.240008,-0.115195,0.205458,0.896890,0.082431,0.268624,...,0.572457,-0.033571,0.499920,0.432836,0.058644,-0.033571,0.807356,0.809466,0.541338,0.240008
3,0.335756,0.205458,1.000000,0.901911,0.720820,-0.249720,1.000000,0.191752,-0.133073,-0.142172,...,0.133537,-0.062127,0.228393,0.105015,0.602835,-0.062127,0.328801,0.388092,0.046598,0.720820
4,0.223952,0.101578,0.901911,1.000000,0.552770,-0.208393,0.901911,0.081395,-0.224368,-0.212328,...,0.114096,-0.044692,0.093240,0.276627,0.783091,-0.044692,0.207139,0.293768,-0.018084,0.552770
5,0.402489,0.240008,0.720820,0.552770,1.000000,-0.171977,0.720820,0.218391,-0.063191,-0.072168,...,0.095936,-0.088446,0.356908,-0.022526,0.685815,-0.088446,0.434974,0.473839,0.050434,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193579,-0.021795,-0.033571,-0.062127,-0.044692,-0.088446,-0.032153,-0.062127,-0.022402,-0.051561,-0.056529,...,0.051675,1.000000,-0.072543,0.059406,-0.055951,1.000000,-0.034474,-0.017719,0.017998,-0.088446
193581,0.905816,0.807356,0.328801,0.207139,0.434974,0.050850,0.328801,0.657862,0.276359,0.324624,...,0.736334,-0.034474,0.732025,0.574082,0.215977,-0.034474,1.000000,0.941512,0.785419,0.434974
193583,0.930034,0.809466,0.388092,0.293768,0.473839,-0.166245,0.388092,0.631067,-0.063661,0.052869,...,0.776608,-0.017719,0.573638,0.643078,0.306020,-0.017719,0.941512,1.000000,0.642582,0.473839


In [267]:
mat3 = cosine_similarity(get_year_item_user_matrix(x_train))
mat3 = pd.DataFrame(mat3, index=movieid_idx, columns=movieid_idx)
mat3

movieId,1,2,3,4,5,6,7,8,9,10,...,190219,190221,193565,193567,193571,193579,193581,193583,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.319163,0.241982,0.201848,0.198905,0.211023,0.111220,0.063820,0.063820,0.072668,0.369299
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.319163,0.241982,0.201848,0.198905,0.211023,0.111220,0.063820,0.063820,0.072668,0.369299
3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.319163,0.241982,0.201848,0.198905,0.211023,0.111220,0.063820,0.063820,0.072668,0.369299
4,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.319163,0.241982,0.201848,0.198905,0.211023,0.111220,0.063820,0.063820,0.072668,0.369299
5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.319163,0.241982,0.201848,0.198905,0.211023,0.111220,0.063820,0.063820,0.072668,0.369299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193579,0.111220,0.111220,0.111220,0.111220,0.111220,0.111220,0.111220,0.111220,0.111220,0.111220,...,0.166838,0.211958,0.271033,0.441805,0.236815,1.000000,0.319672,0.319672,0.262923,0.067878
193581,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,...,0.149737,0.151474,0.242144,0.239858,0.243532,0.319672,1.000000,1.000000,0.414957,0.054776
193583,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,0.063820,...,0.149737,0.151474,0.242144,0.239858,0.243532,0.319672,1.000000,1.000000,0.414957,0.054776


In [268]:
total_sim_matrix = mat1.add(mat2.add(mat3)).div(3)

total_sim_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,190219,190221,193565,193567,193571,193579,193581,193583,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.692470,0.468147,0.411371,0.491456,0.294975,0.474127,0.637101,0.355339,0.417314,...,0.364600,0.073396,0.278540,0.272766,0.138899,0.029808,0.323212,0.331285,0.254207,0.257263
2,0.692470,1.000000,0.501992,0.368066,0.473499,0.320369,0.431562,0.618897,0.371647,0.451654,...,0.297207,0.069470,0.233923,0.210580,0.089889,0.025883,0.290392,0.291095,0.204669,0.203102
3,0.468147,0.501992,1.000000,0.668763,0.632165,0.340681,0.705630,0.473310,0.324419,0.299997,...,0.150900,0.059952,0.143414,0.101307,0.271286,0.016364,0.130874,0.150637,0.039755,0.363373
4,0.411371,0.368066,0.668763,1.000000,0.507314,0.264857,0.608133,0.411233,0.258544,0.230125,...,0.144420,0.065763,0.098363,0.158510,0.331371,0.022176,0.090320,0.119196,0.018195,0.307356
5,0.491456,0.473499,0.632165,0.507314,1.000000,0.288972,0.628720,0.451876,0.321869,0.356907,...,0.138367,0.051179,0.186252,0.058793,0.298946,0.007591,0.166265,0.179220,0.041034,0.456433
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193579,0.029808,0.025883,0.016364,0.022176,0.007591,0.026356,0.016364,0.029606,0.019886,0.018230,...,0.072838,0.403986,0.399497,0.500404,0.393621,1.000000,0.428399,0.433984,0.426974,-0.006856
193581,0.323212,0.290392,0.130874,0.090320,0.166265,0.038223,0.130874,0.240561,0.113393,0.129481,...,0.295357,0.039000,0.658056,0.604647,0.486503,0.428399,1.000000,0.980504,0.733459,0.163250
193583,0.331285,0.291095,0.150637,0.119196,0.179220,-0.034142,0.150637,0.231629,0.000053,0.038896,...,0.308782,0.044585,0.605261,0.627645,0.516517,0.433984,0.980504,1.000000,0.685847,0.176205


In [269]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
movieId,,,,,,,,,,,,,,,,,,,,,
1,-0.457947,NaN,NaN,NaN,0.367146,NaN,0.954981,NaN,NaN,NaN,...,-0.935911,NaN,0.426517,-0.644222,0.964425,-1.598352,0.221511,-0.587601,-0.6003,1.52952
2,NaN,NaN,NaN,NaN,NaN,0.595275,NaN,0.437643,NaN,NaN,...,NaN,0.621013,NaN,2.040036,0.353715,NaN,NaN,-1.050881,NaN,NaN
3,-0.457947,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.050881,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,-0.580300,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,1.770850,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.644222,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193585,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [270]:
genre_year_sim_matrix

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.992201,0.937751,0.939113,0.942758,0.900123,0.937751,0.982575,0.893879,0.933102,...,0.250952,0.284705,0.216505,0.264727,0.082266,0.204666,0.243542,0.042353,0.194310,0.371159
2,0.992201,1.000000,0.917556,0.920493,0.920617,0.889105,0.917556,0.985755,0.889646,0.930949,...,0.242261,0.254766,0.199724,0.222961,0.076510,0.198745,0.232152,0.031141,0.164454,0.348095
3,0.937751,0.917556,1.000000,0.991114,0.976337,0.905018,1.000000,0.908562,0.889149,0.918604,...,0.200273,0.185155,0.255940,0.084693,0.067052,0.063976,0.089834,0.071636,0.005915,0.375759
4,0.939113,0.920493,0.991114,1.000000,0.965673,0.930860,0.991114,0.911524,0.900818,0.934912,...,0.178628,0.198690,0.260956,0.071543,0.074925,0.032585,0.055232,0.140303,-0.015937,0.354718
5,0.942758,0.920617,0.976337,0.965673,1.000000,0.900568,0.976337,0.909629,0.887640,0.914038,...,0.219389,0.165997,0.270785,0.090914,0.068291,0.100089,0.127486,0.027023,0.023113,0.403320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193581,0.204666,0.198745,0.063976,0.032585,0.100089,0.038372,0.063976,0.175210,0.110720,0.082147,...,0.405170,0.420108,0.207357,0.592009,0.252292,1.000000,0.965138,0.453875,0.610263,0.114300
193583,0.243542,0.232152,0.089834,0.055232,0.127486,0.003399,0.089834,0.197103,0.039002,0.046048,...,0.362006,0.462037,0.221197,0.648677,0.224709,0.965138,1.000000,0.408236,0.576274,0.138270
193585,0.042353,0.031141,0.071636,0.140303,0.027023,0.114799,0.071636,0.034465,0.051231,0.067910,...,0.112636,0.328660,0.320669,0.152490,0.264391,0.453875,0.408236,1.000000,0.023796,-0.038360


In [271]:
def RMSE(y_true, y_pred):
    return np.sqrt(np.mean( (np.array(y_true) - np.array(y_pred)) **2))

def score(model, item_user_matrix, sim_matrix, is_centered):
    id_pairs = zip(x_test['userId'], x_test['movieId']) # x_test : movie_ratings_df
    
    y_pred = np.array([model(user, movie, item_user_matrix, sim_matrix, is_centered) for (user, movie) in id_pairs])
    y_true = np.array(x_test['rating'])
    return RMSE(y_true, y_pred)

In [272]:
def ib_cf(user_id, movie_id, item_user_matrix, sim_matrix, is_centered): # model
    if movie_id in sim_matrix:
        sim_scores = sim_matrix[movie_id]
        
        user_rating = item_user_matrix[user_id]
        non_rating_idx = user_rating[user_rating.isnull()].index
        
        user_rating = user_rating.dropna()
        sim_scores = sim_scores.drop(index=non_rating_idx)
        if is_centered:
            deviation = np.dot(sim_scores, user_rating) / sim_scores.sum()
            mean = user_rating_mean[user_id]
            mean_rating = mean+deviation
        else:
            mean_rating = np.dot(sim_scores, user_rating) / sim_scores.sum()
    else:
        mean_rating=3.0
    return mean_rating

In [273]:
genre_year_sim_matrix = cosine_similarity(get_genre_item_user_matrix(x_train).add(get_year_item_user_matrix(x_train)))
genre_year_sim_matrix = pd.DataFrame(genre_year_sim_matrix, index=movieid_idx, columns=movieid_idx)
no_centered_item_user_matrix = x_train.pivot_table(values='rating', index='movieId', columns='userId')
score(ib_cf, no_centered_item_user_matrix, genre_year_sim_matrix, is_centered=False)

0.9323264462453875

In [274]:
weighted_item_user_matrix = x_train.pivot_table(values='weighted_centered_rating', index='movieId', columns='userId')
score(ib_cf, weighted_item_user_matrix, sim_matrix=total_sim_matrix, is_centered=True)

1.189767390584796

In [275]:
item_user_matrix = x_train.pivot_table(values='centered_rating', index='movieId', columns='userId')
score(ib_cf, item_user_matrix, sim_matrix=total_sim_matrix, is_centered=True)

0.9190520124480015